<a href="https://colab.research.google.com/github/gmonitillodev-hub/ProfessionAI/blob/main/Es2_Python_Concourrency.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import threading # Used for creating and managing threads
import multiprocessing # Used for creating and managing processes
import time # Used for time-related functions (e.g., sleep)
import random # Used for generating random numbers (e.g., sleep duration)
from queue import Queue # Although imported, this is not directly used for the shared queue due to multiprocessing.Manager

# Simulazione di un centro assistenza con più operatori

# Global parameters for the simulation
NUM_OPERATORI = 3   # Maximum number of operators (threads) available to handle requests concurrently
TEMPO_MIN = 2       # Minimum time (in seconds) an operator or worker takes to handle a request
TEMPO_MAX = 5       # Maximum time (in seconds) an operator or worker takes to handle a request
NUM_PROCESSI = 2    # Number of processes (workers) running in multiprocessing
NUM_RICHIESTE = 10  # Total number of customer requests to be generated

# Initialize a manager to create shareable objects that can be accessed by both threads and processes
manager = multiprocessing.Manager()
coda_richieste = manager.Queue() # A shared queue to hold customer requests, accessible by all workers
# A lock to synchronize print statements across threads and processes to prevent interleaved output
print_lock = manager.Lock()

# Semaphore for controlling the number of concurrent *thread* operators that can process requests
semaphore_thread_operators = threading.Semaphore(NUM_OPERATORI)


def genera_richieste(coda_richieste, num_richieste, print_lock):
    """Generates customer requests and puts them into the shared queue."""
    for i in range(1, num_richieste + 1):
        cliente = f"Cliente-{i}"
        with print_lock:
            print(f"Generatore: {cliente} ha inviato una richiesta e messa in coda.")
        coda_richieste.put(cliente) # Add the customer request to the queue
        time.sleep(random.uniform(0.5, 1.5)) # Simulate varying time between requests


def gestisci_richiesta_thread(coda_richieste, semaphore_thread_operators, print_lock):
    """Thread function to handle customer requests. Simulates an operator."""
    while True:
        semaphore_thread_operators.acquire() # Acquire a semaphore slot, limiting concurrent thread operators
        cliente = coda_richieste.get() # Get a request from the shared queue

        if cliente is None: # Check for the 'poison pill' to signal termination
            coda_richieste.put(None) # Propagate the poison pill for other workers
            semaphore_thread_operators.release() # Release the semaphore slot
            break # Exit the loop, terminating the thread

        with print_lock:
            print(f"Operatore (thread {threading.current_thread().name}): sta gestendo la richiesta di {cliente}...")
        time.sleep(random.randint(TEMPO_MIN, TEMPO_MAX)) # Simulate work being done
        with print_lock:
            print(f"Operatore (thread {threading.current_thread().name}): ha assistito {cliente}.")
        semaphore_thread_operators.release() # Release the semaphore slot after handling the request


def worker_multiprocessing(coda_richieste, print_lock):
    """Process function to handle customer requests. Simulates a backend worker."""
    while True:
        cliente = coda_richieste.get() # Get a request from the shared queue

        if cliente is None: # Check for the 'poison pill' to signal termination
            coda_richieste.put(None) # Propagate the poison pill for other workers
            break # Exit the loop, terminating the process

        with print_lock:
            print(
                f"Processo {multiprocessing.current_process().name}: "
                f"sta gestendo la richiesta di {cliente}..."
            )
        time.sleep(random.randint(TEMPO_MIN, TEMPO_MAX)) # Simulate work being done
        with print_lock:
            print(
                f"Processo {multiprocessing.current_process().name}: "
                f"ha assistito {cliente}."
            )


def avvia_centro_assistenza(num_richieste):
    """Orchestrates the start and stop of request generation, thread operators, and multiprocessing workers."""
    with print_lock:
        print("\nCentro gestione in avvio...\n")

    # Start the request generator thread
    thread_generatore = threading.Thread(
        target=genera_richieste,
        args=(coda_richieste, num_richieste, print_lock),
        name="GeneratoreRichieste"
    )
    thread_generatore.start()
    # REMOVED: thread_generatore.join() # This was preventing concurrency before workers start

    # Start operator threads (NUM_OPERATORI threads)
    threads_operatori = [
        threading.Thread(
            target=gestisci_richiesta_thread,
            args=(coda_richieste, semaphore_thread_operators, print_lock),
            name=f"Thread-Operatore-{i+1}"
        )
        for i in range(NUM_OPERATORI)
    ]
    for t in threads_operatori:
        t.start()

    # Start multiprocessing workers (NUM_PROCESSI processes)
    processi = [
        multiprocessing.Process(
            target=worker_multiprocessing,
            args=(coda_richieste, print_lock),
            name=f"Processo-Worker-{i+1}"
        )
        for i in range(NUM_PROCESSI)
    ]
    for p in processi:
        p.start()

    # Wait for the request generation thread to finish creating all requests
    # This ensures all requests are in the queue before signaling workers to stop.
    thread_generatore.join()

    # Add 'poison pills' to the queue to signal all worker threads and processes to stop.
    # Each worker (thread or process) will consume one 'None' and put it back for the next worker.
    # The total number of poison pills should match the total number of active workers.
    for _ in range(NUM_OPERATORI + NUM_PROCESSI):
        coda_richieste.put(None)

    # Wait for all operator threads to finish their execution (after consuming their poison pill)
    for t in threads_operatori:
        t.join()

    # Wait for all multiprocessing workers to finish their execution (after consuming their poison pill)
    for p in processi:
        p.join()

    with print_lock:
        print("\nTutte le richieste sono state gestite con successo\n")


if __name__ == "__main__":
    # Entry point of the script, starts the simulation
    avvia_centro_assistenza(NUM_RICHIESTE)


Centro gestione in avvio...

Generatore: Cliente-1 ha inviato una richiesta e messa in coda.
Generatore: Cliente-2 ha inviato una richiesta e messa in coda.
Generatore: Cliente-3 ha inviato una richiesta e messa in coda.
Generatore: Cliente-4 ha inviato una richiesta e messa in coda.
Generatore: Cliente-5 ha inviato una richiesta e messa in coda.
Generatore: Cliente-6 ha inviato una richiesta e messa in coda.
Generatore: Cliente-7 ha inviato una richiesta e messa in coda.
Generatore: Cliente-8 ha inviato una richiesta e messa in coda.
Generatore: Cliente-9 ha inviato una richiesta e messa in coda.
Generatore: Cliente-10 ha inviato una richiesta e messa in coda.
Operatore (thread Thread-Operatore-2): sta gestendo la richiesta di Cliente-1...
Operatore (thread Thread-Operatore-3): sta gestendo la richiesta di Cliente-2...
Operatore (thread Thread-Operatore-2): sta gestendo la richiesta di Cliente-1...
Processo Processo-Worker-1: sta gestendo la richiesta di Cliente-4...
Operatore (threa

## Lock e Semaphore

Il `Lock` e il `Semaphore` sono strumenti utilizzati per coordinare l’esecuzione concorrente di thread e processi.

### Lock

Il `Lock` garantisce che una sezione di codice possa essere eseguita da un solo worker alla volta.

```python
with print_lock:
    print("Gestione richiesta")
```

Quando un worker acquisisce il lock, gli altri devono attendere che venga rilasciato. Il blocco `with` gestisce automaticamente l’acquisizione e il rilascio del lock, anche in caso di errore.

Nel centro assistenza, `print_lock` impedisce che thread e processi scrivano contemporaneamente sulla console, evitando messaggi sovrapposti.

Poiché il lock è condiviso anche tra processi, viene creato tramite il manager:

```python
print_lock = manager.Lock()
```

### Semaphore

Il `Semaphore` limita il numero massimo di worker che possono accedere contemporaneamente a una determinata operazione.

```python
semaphore = threading.Semaphore(3)
```

In questo esempio sono disponibili tre permessi. Ogni thread deve acquisirne uno prima di iniziare:

```python
semaphore.acquire()
```

Al termine deve restituirlo:

```python
semaphore.release()
```

Se tutti i permessi sono occupati, gli altri thread rimangono in attesa.

Può essere utilizzato anche tramite `with`:

```python
with semaphore:
    gestisci_richiesta()
```

### Differenza principale

| Strumento      | Accessi simultanei | Scopo                                                  |
| -------------- | -----------------: | ------------------------------------------------------ |
| `Lock`         |                  1 | Proteggere una risorsa condivisa o una sezione critica |
| `Semaphore(n)` |                `n` | Limitare il numero di operazioni contemporanee         |

In sintesi, il `Lock` garantisce l’accesso esclusivo, mentre il `Semaphore` gestisce un numero configurabile di accessi concorrenti.

Nel codice del centro assistenza, il semaforo è tecnicamente ridondante perché vengono creati esattamente tre thread e il limite è anch’esso pari a tre. Diventerebbe utile, ad esempio, creando dieci thread ma consentendo soltanto a tre di gestire richieste contemporaneamente.
